# Business solution Project
## Challenge
Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

In [35]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from google import genai

In [42]:
# Initialize and constants

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY in your .env file.")

MODEL = "gemini-3.5-flash"
client = genai.Client(api_key=api_key)

In [43]:
response = client.interactions.create(
    model=MODEL,
    input="What is AI",
)
print(response.output_text)

**Artificial Intelligence (AI)** refers to computer systems and software capable of performing tasks that typically require human intelligence. These tasks include reasoning, learning from past experiences, making decisions, solving problems, and understanding language.

Instead of following static, pre-written instructions for every situation, AI systems analyze vast amounts of data to find patterns, make predictions, and adapt to new information.

---

### How Does AI Work?

At its core, modern AI relies on data and mathematical algorithms. The primary technologies powering AI today are:

1. **Machine Learning (ML):** A subset of AI where computers are trained to recognize patterns in data without being explicitly programmed for every scenario. 
   * *Example:* Showing a computer 10,000 photos of cats so it learns to identify a cat on its own.
2. **Deep Learning:** An advanced form of Machine Learning that uses **Neural Networks**—complex computational structures inspired by the huma

In [44]:
links = fetch_website_links("https://edwarddonner.com/")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward

In [45]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [46]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [47]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/11/11/ai-sdlc-the-ai-native-software-engineer-in-the-agentic-era/
https://edwarddo

In [48]:
# Define the schema once
LINKS_SCHEMA = {
    "type": "object",
    "properties": {
        "links": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "type": {"type": "string"},
                    "url": {"type": "string"},
                },
                "required": ["type", "url"],
            },
        }
    },
    "required": ["links"],
}

In [49]:
def select_relevant_links(url: str) -> dict:
    response = client.interactions.create(
        model=MODEL,
        input=f"{link_system_prompt}\n\n{get_links_user_prompt(url)}",
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": LINKS_SCHEMA,
        },
    )
    return json.loads(response.output_text)

In [50]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}